# 🇯 Xiangqi-R1 Community Data Miner — JRCP 3.0
## Đóng góp sức mạnh CPU cho hệ sinh thái cờ Tướng AI!

### 🔧 Hướng dẫn (3 bước duy nhất):
1. **Gắn GPU Runtime**: Runtime → Change runtime type → T4 GPU
2. **Cài Secret**: 🔑 icon bên trái → Add new secret → Name: `HF_TOKEN`, Value: [token của bạn](https://huggingface.co/settings/tokens)
3. **Chạy tất cả**: Runtime → Run all (Ctrl+F9)

### ✅ Mọi thứ khác tự động:
- Tự cài Rust toolchain + biên dịch engine (~3 phút)
- Tự mine dữ liệu depth 12 chất lượng cao (JRCP 3.0 — 14 chiều kích phân tích)
- Tự push lên HuggingFace mỗi 5 phút
- Tự lưu checkpoint — nếu Colab hết giờ, chạy lại Run All sẽ tiếp tục
- Tự dọn dẹp khi bị ngắt (Graceful Shutdown)

---
### 🌏 For English speakers:
1. **Attach GPU**: Runtime → Change runtime type → T4 GPU
2. **Set Secret**: 🔑 icon on left → Add secret → Name: `HF_TOKEN`, Value: [your token](https://huggingface.co/settings/tokens)
3. **Run All**: Runtime → Run all (Ctrl+F9)

Everything else is automated. Your CPU power helps train better Chinese Chess AI!

In [ ]:
# === XIANGQI-R1 COMMUNITY JRCP 3.0 DATA MINER — AUTO SETUP ===
import os, sys, signal, json, time, subprocess, hashlib, glob, shutil
from pathlib import Path
from datetime import datetime

print("🇯 Xiangqi-R1 Community JRCP 3.0 Data Miner v3.0")
print("=" * 60)

# ── 1. GPU CHECK ──
print("\n🔍 [1/5] Kiểm tra GPU...")
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
    if gpu.returncode == 0:
        print(f"   ✅ GPU: {gpu.stdout.strip()}")
    else:
        print("   ⚠️ Không phát hiện GPU — Mining sẽ dùng CPU")
except FileNotFoundError:
    print("   ⚠️ nvidia-smi không có — Mining sẽ dùng CPU")

# ── 2. HF TOKEN (Colab Secrets → os.environ → input) ──
print("\n🔑 [2/5] Xác thực HuggingFace Token...")
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print("   ✅ HF_TOKEN từ Colab Secrets")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("   ✅ HF_TOKEN từ os.environ")
if not HF_TOKEN:
    print("   ⚠️ Không tìm thấy HF_TOKEN!")
    print("   💡 Hướng dẫn: Click 🔑 icon bên trái → Add Secret → Name: HF_TOKEN")
    HF_TOKEN = input("   Nhập HF_TOKEN: ").strip()
assert HF_TOKEN and len(HF_TOKEN) > 10, "\n❌ Cần HF_TOKEN hợp lệ! Xem hướng dẫn ở Cell 0."
os.environ['HF_TOKEN'] = HF_TOKEN
print(f"   Token: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")

# ── 3. INSTALL PYTHON DEPENDENCIES ──
print("\n📦 [3/5] Cài đặt dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'],
               capture_output=True)
from huggingface_hub import HfApi, hf_hub_download
print("   ✅ huggingface_hub installed")

# ── 4. CLONE/PULL REPO ──
print("\n📁 [4/5] Tải mã nguồn engine...")
REPO_DIR = '/content/xiangqi-rim'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, capture_output=True)
    print(f"   ✅ Git pull thành công: {REPO_DIR}")
else:
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://huggingface.co/spaces/hoduyquocbao/xiangqi-rim', REPO_DIR],
                   capture_output=True)
    print(f"   ✅ Git clone thành công: {REPO_DIR}")

# ── 5. INSTALL RUST TOOLCHAIN ──
print("\n🧰 [5/5] Cài đặt Rust toolchain...")
rust_check = subprocess.run(['cargo', '--version'], capture_output=True)
if rust_check.returncode != 0:
    print("   ⚙️ Đang cài Rust (~1 phút)...")
    subprocess.run('curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y',
                   shell=True, capture_output=True)
    os.environ['PATH'] = f"{Path.home()}/.cargo/bin:" + os.environ['PATH']
rust_ver = subprocess.check_output(['cargo', '--version']).decode().strip()
print(f"   ✅ {rust_ver}")

print("\n" + "=" * 60)
print("✅ SETUP HOÀN TẤT! Tiếp tục với Cell tiếp theo...")
print("=" * 60)

In [ ]:
# === BIÊN DỊCH JRCP 3.0 × 64GB RAM MINER ===
import subprocess, time

REPO_DIR = '/content/xiangqi-rim'
print("⚙️ Biên dịch Native Rust Engine (~3 phút lần đầu, ~5s lần sau)...")
start = time.time()

result = subprocess.run(
    ['cargo', 'build', '--release', '--example', '23_jrcp3_ram64g_miner'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True
)

elapsed = time.time() - start
if result.returncode == 0:
    print(f"✅ Biên dịch thành công trong {elapsed:.1f}s!")
else:
    print(f"❌ Lỗi biên dịch:\n{result.stderr[-500:]}")
    raise RuntimeError("Không thể biên dịch engine. Vui lòng báo lỗi.")

In [ ]:
# @title ⚙️ Cấu Hình Mining (Chỉnh theo ý muốn) { display-mode: "form" }
import time

GAMES = 200  # @param {type:"slider", min:50, max:5000, step:50}
DEPTH = 12  # @param {type:"slider", min:4, max:16, step:2}
SEED = int(time.time()) % 100000  # @param {type:"integer"}
PUSH_INTERVAL_SEC = 300  # @param {type:"slider", min:60, max:900, step:60}
DATASET_REPO = "hoduyquocbao/xiangqi-nnue-dataset"  # @param {type:"string"}

# Tính toán ETA
speed_estimate = {4: 40.0, 6: 10.0, 8: 2.0, 10: 0.6, 12: 0.3, 14: 0.1, 16: 0.03}
speed = speed_estimate.get(DEPTH, 1.0)
positions = GAMES * 50  # ~50 vị trí/ván
eta_seconds = positions / max(speed, 0.01)
eta_hours = eta_seconds / 3600

print(f"🎯 Cấu hình Mining:")
print(f"   Games    : {GAMES:,}")
print(f"   Depth    : {DEPTH}")
print(f"   Seed     : {SEED}")
print(f"   Push     : mỗi {PUSH_INTERVAL_SEC}s ({PUSH_INTERVAL_SEC // 60} phút)")
print(f"   Dataset  : {DATASET_REPO}")
print(f"   \n📊 Ước tính: ~{positions:,} vị trí @ {speed:.1f} FEN/s")
print(f"   ⏱️ ETA: {eta_hours:.1f} giờ ({eta_seconds / 60:.0f} phút)")

if eta_hours > 11:
    sessions = int(eta_hours / 11) + 1
    print(f"   ⚠️ Cần ~{sessions} phiên Colab. Hệ thống sẽ tự checkpoint + resume!")

In [ ]:
# === MINING CORE — Auto Push + Graceful Shutdown + Checkpoint ===
import os, sys, signal, json, time, subprocess, threading
from datetime import datetime
from huggingface_hub import HfApi

REPO_DIR = '/content/xiangqi-rim'
BINARY = f"{REPO_DIR}/target/release/examples/23_jrcp3_ram64g_miner"
OUTPUT = f"/content/jrcp3_d{DEPTH}_s{SEED}_{int(time.time())}.jsonl"
CHECKPOINT = f"/content/checkpoint_d{DEPTH}_s{SEED}.json"

# ── Resume từ checkpoint nếu có ──
resume_games = 0
if os.path.exists(CHECKPOINT):
    try:
        ckpt = json.load(open(CHECKPOINT))
        resume_games = ckpt.get('completed', 0)
        if ckpt.get('status') == 'done':
            print(f"✅ Phiên trước đã hoàn tất {resume_games} games. Bắt đầu phiên mới.")
            resume_games = 0
        else:
            print(f"🔄 Resume từ checkpoint: {resume_games} games đã hoàn thành")
            # Tải output từ phiên trước nếu còn
            prev_output = ckpt.get('output', '')
            if os.path.exists(prev_output):
                OUTPUT = prev_output
                print(f"   Tiếp tục ghi vào: {OUTPUT}")
    except Exception:
        pass

effective_games = max(0, GAMES - resume_games)
if effective_games == 0:
    print("✅ Đã hoàn tất số games yêu cầu!")
    # sys.exit(0)  # Don't exit in notebook
else:
    print(f"\n🏭 Mining {effective_games} games @ depth {DEPTH}, seed {SEED}")
    print(f"☁️ Auto-push mỗi {PUSH_INTERVAL_SEC}s ({PUSH_INTERVAL_SEC // 60} phút)")
    print(f"💾 Checkpoint: {CHECKPOINT}")
    print(f"📄 Output: {OUTPUT}")
    print("\n" + "─" * 60)

    # ── HF API ──
    api = HfApi(token=os.environ['HF_TOKEN'])
    push_count = 0

    def push_to_hub(filepath, reason="periodic"):
        """Push file lên HuggingFace Dataset Hub."""
        global push_count
        if not os.path.exists(filepath) or os.path.getsize(filepath) == 0:
            return 0
        try:
            fname = os.path.basename(filepath)
            api.upload_file(
                path_or_fileobj=filepath,
                path_in_repo=f"community/{fname}",
                repo_id=DATASET_REPO,
                repo_type="dataset",
                commit_message=f"[Community-Mine] {reason} | d{DEPTH} s{SEED} | {datetime.now().strftime('%Y%m%d_%H%M')}"
            )
            lines = sum(1 for _ in open(filepath))
            push_count += 1
            print(f"   ☁️ Push #{push_count}: {lines:,} mẫu → {DATASET_REPO}/community/{fname}")
            return lines
        except Exception as e:
            print(f"   ⚠️ Push lỗi (sẽ thử lại lần sau): {str(e)[:100]}")
            return 0

    def save_checkpoint(completed, status='running'):
        """Lưu checkpoint để resume phiên sau."""
        json.dump({
            'completed': completed,
            'seed': SEED,
            'depth': DEPTH,
            'output': OUTPUT,
            'status': status,
            'timestamp': datetime.now().isoformat()
        }, open(CHECKPOINT, 'w'), indent=2)

    # ── Graceful Shutdown Handler ──
    shutdown_requested = False
    current_games = [0]  # Mutable in closure

    def graceful_shutdown(signum, frame):
        nonlocal shutdown_requested
        shutdown_requested = True
        print(f"\n\n🛡️ {'SIGTERM' if signum == signal.SIGTERM else 'SIGINT'} nhận được!")
        print("   Đang flush + push dữ liệu...")
        push_to_hub(OUTPUT, reason="graceful_shutdown")
        save_checkpoint(resume_games + current_games[0], status='interrupted')
        print("   💾 Checkpoint đã lưu. Dữ liệu an toàn!")
        print("   🔁 Chạy lại Runtime → Run All để tiếp tục.")

    signal.signal(signal.SIGTERM, graceful_shutdown)
    signal.signal(signal.SIGINT, graceful_shutdown)

    # ── Khởi chạy Mining Process ──
    env = {
        **os.environ,
        'GAMES': str(effective_games),
        'DEPTH': str(DEPTH),
        'THREADS': '2',
        'TT_MB': '512',
        'SIEVE_MB': '4096',
        'SEED': str(SEED + resume_games),  # Offset seed để không lặp dữ liệu
        'OUTPUT': OUTPUT,
    }

    proc = subprocess.Popen(
        [BINARY], env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        bufsize=1
    )

    last_push_time = time.time()
    mining_start = time.time()

    try:
        for line in proc.stdout:
            line = line.rstrip()
            if not line:
                continue

            # Hiển thị output
            print(line)

            # Parse progress từ log
            if 'MINING STREAMING' in line:
                try:
                    parts = line.split('/')
                    current_games[0] = int(parts[0].split()[-1])
                except (ValueError, IndexError):
                    pass

            # Auto-push định kỳ
            elapsed_since_push = time.time() - last_push_time
            if elapsed_since_push >= PUSH_INTERVAL_SEC:
                pushed = push_to_hub(OUTPUT, reason=f"periodic_g{current_games[0]}")
                if pushed > 0:
                    save_checkpoint(resume_games + current_games[0])
                last_push_time = time.time()

            if shutdown_requested:
                proc.terminate()
                break

    except KeyboardInterrupt:
        graceful_shutdown(signal.SIGINT, None)
        proc.terminate()

    proc.wait()

    # ── Final Push ──
    total_elapsed = time.time() - mining_start
    print(f"\n{'=' * 60}")
    print(f"🏁 Mining kết thúc sau {total_elapsed / 60:.1f} phút")
    push_to_hub(OUTPUT, reason="completed")
    save_checkpoint(resume_games + current_games[0], status='done')
    print(f"💾 Checkpoint final đã lưu.")

In [ ]:
# === KẾT QUẢ & CẢM ƠN ===
import os, json

OUTPUT_FILES = [f for f in [OUTPUT] if os.path.exists(f) and os.path.getsize(f) > 0]

if OUTPUT_FILES:
    for f in OUTPUT_FILES:
        total_lines = sum(1 for _ in open(f))
        size_mb = os.path.getsize(f) / 1024 / 1024

        # Kiểm tra JSON validity
        valid = 0
        invalid = 0
        for line in open(f):
            try:
                json.loads(line.strip())
                valid += 1
            except (json.JSONDecodeError, Exception):
                invalid += 1

        print(f"{'=' * 60}")
        print(f"  🏆 CẢM ƠN BẠN ĐÃ ĐÓNG GÓP!")
        print(f"  📊 Tổng mẫu: {total_lines:,} ({valid:,} hợp lệ, {invalid} lỗi)")
        print(f"  📁 Kích thước: {size_mb:.1f} MB")
        print(f"  🎯 Depth: {DEPTH}")
        print(f"  🌱 Seed: {SEED}")
        print(f"  ☁️ Dataset: {DATASET_REPO}")
        print(f"{'=' * 60}")
        print(f"\n  Mỗi mẫu bạn đóng góp giúp Xiangqi-R1 thông minh hơn!")
        print(f"  🔁 Chạy lại Runtime → Run All để tiếp tục đóng góp.")
        print(f"  ❤️ Thank you for contributing to open-source Chinese Chess AI!")
else:
    print("⚠️ Không tìm thấy dữ liệu mining. Vui lòng chạy Cell 4 trước.")